In [1]:
from pathlib import Path
import logging
import joblib
import numpy as np
from tqdm import tqdm
from src.module.utils import load_config, process_meanbp
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
)
import matplotlib.pyplot as plt

In [2]:
def extract_lastmbp(arr, config):
    mbp = process_meanbp(arr.reshape(-1), config)
    mbp = np.unique(mbp[np.where(mbp != 0)])
    return mbp[-1]

Methods 표현은 "모든 모델의 threshold는 development set(train+validation)에서 specificity 80/90/95%에 해당하는 지점으로 고정한 뒤 test에 적용하였다"로 통일하시면 됩니다. 딥러닝·머신러닝 모델도 같은 기준을 쓰면 비교가 일관됩니다.

In [3]:
config = load_config("../config/conf.yaml")
src_path = Path(config["data_path"]).expanduser() / "vitaldb" / "06.train_val_test_new"
model_path = Path(config["model_path"]).expanduser() / "revision"
strategy = "hypophetversion2"

In [4]:
train_X = np.load(src_path / f"vitaldb_{strategy}_train_X.npy")
train_y = np.load(src_path / f"vitaldb_{strategy}_train_y.npy")
val_X = np.load(src_path / f"vitaldb_{strategy}_val_X.npy")
val_y = np.load(src_path / f"vitaldb_{strategy}_val_y.npy")
test_X = np.load(src_path / f"vitaldb_{strategy}_test_X.npy")
test_y = np.load(src_path / f"vitaldb_{strategy}_test_y.npy")
print(train_X.shape, train_y.shape)
print(val_X.shape, val_y.shape)
print(test_X.shape, test_y.shape)

(145764, 3000, 1) (145764,)
(37159, 3000, 1) (37159,)
(44056, 3000, 1) (44056,)


In [5]:
trval_X = np.vstack([train_X, val_X])
trval_y = np.hstack([train_y, val_y])
print(trval_X.shape, trval_y.shape)

(182923, 3000, 1) (182923,)


In [6]:
# threshold 구하기 목적
# lastmbp_trval = np.array([extract_lastmbp(x, config) for x in tqdm(trval_X, ncols=75)])
lastmbp_trval = np.array([extract_lastmbp(x, config) for x in tqdm(val_X, ncols=75)])
print(lastmbp_trval.shape)

100%|██████████████████████████████| 37159/37159 [00:06<00:00, 6107.59it/s]

(37159,)


In [8]:
neg = lastmbp_trval[np.where(val_y == 0)]

In [9]:
for spec in [0.8, 0.9, 0.95]:
    cutoff = np.quantile(neg, 1 - spec)  # negative 중에 1-spec% 만 양성으로 판별
    print(spec, cutoff)

0.8 80.36432800292968
0.9 76.37737579345703
0.95 73.67890930175781


In [13]:
# eval 목적
lastmbp_test = np.array([extract_lastmbp(x, config) for x in tqdm(test_X, ncols=75)])
print(lastmbp_test.shape)

100%|██████████████████████████████| 44056/44056 [00:13<00:00, 3366.85it/s]

(44056,)


In [14]:
# 원래는 1에 가까울수록 양성, 0에 가까울수록 음성이다.
# 근데 mbp의 경우 값이 커질수록 음성, 반대는 양성이다.
# 그러므로 이를 반전시켜준다.
#
# 확률 출력이 없어 보정 지표 (BS, BSS, Calib)는 산출하지 않았다
auroc = roc_auc_score(test_y, -lastmbp_test)
auprc = average_precision_score(test_y, -lastmbp_test)
print(f"AUROC: {auroc}")
print(f"AUPRC: {auprc}")

AUROC: 0.7436351241401977
AUPRC: 0.6387655069822091


In [18]:
print("Specificity: 0.8")
print(classification_report(test_y, np.where(lastmbp_test < 81.10087432861327, 1, 0)))

Specificity: 0.8
              precision    recall  f1-score   support

         0.0       0.78      0.79      0.79     28569
         1.0       0.60      0.58      0.59     15487

    accuracy                           0.72     44056
   macro avg       0.69      0.69      0.69     44056
weighted avg       0.72      0.72      0.72     44056



In [ ]:
print("Specificity: 0.9")
print(classification_report(test_y, np.where(lastmbp_test < 76.90589370727538, 1, 0)))

Specificity: 0.9
              precision    recall  f1-score   support

         0.0       0.74      0.90      0.81     28569
         1.0       0.69      0.42      0.52     15487

    accuracy                           0.73     44056
   macro avg       0.71      0.66      0.67     44056
weighted avg       0.72      0.73      0.71     44056



In [20]:
print("Specificity: 0.95")
print(classification_report(test_y, np.where(lastmbp_test < 74.11129112243653, 1, 0)))

Specificity: 0.95
              precision    recall  f1-score   support

         0.0       0.71      0.95      0.81     28569
         1.0       0.74      0.28      0.41     15487

    accuracy                           0.71     44056
   macro avg       0.73      0.62      0.61     44056
weighted avg       0.72      0.71      0.67     44056

